In [29]:
# OPRO for Clause Classification - Full Notebook
import pandas as pd
import os
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score

load_dotenv()
import openai

# Configure OpenRouter as OpenAI-compatible endpoint
openai.api_key = os.getenv("OPENROUTER_API_KEY")
openai.api_base = "https://openrouter.ai/api/v1"

# Load datasets
train_df = pd.read_csv("claudette_train_merged.tsv", sep="\t")
val_df = pd.read_csv("claudette_val_merged.tsv", sep="\t")
test_df = pd.read_csv("claudette_test_merged.tsv", sep="\t")


In [30]:
def build_meta_prompt(exemplars, solutions):
    prompt = "You are optimizing a classifier prompt. Your goal is to generate a new instruction that improves clause classification accuracy.\n\n"
    
    prompt += "## Training Examples:\n"
    for _, row in exemplars.iterrows():
        prompt += f"Clause: {row['text']}\nLabel: {row['label']}\n\n"
    
    prompt += "## Previous Instructions and Accuracy:\n"
    for text, score in solutions:
        prompt += f"Instruction: [{text}]\nScore: {score}\n\n"
    
    prompt += "## Task:\nGenerate a new instruction that is different from the previous ones and likely to improve accuracy. Respond with the instruction in square brackets only."
    return prompt


In [31]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def get_instruction(prompt, model="mistralai/mistral-7b-instruct", temperature=1.0):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip("[] \n")


def score_instruction(instruction, exemplars, val_df, model="mistralai/mistral-7b-instruct", sample_size=20, verbose=True):
    prompt_prefix = instruction + "\n\n"
    preds = []

    val_sample = val_df.sample(sample_size, random_state=42) if sample_size < len(val_df) else val_df

    if verbose:
        print(f"\n🧪 Scoring instruction: \"{instruction}\"")
        print(f"Using model: {model} on {len(val_sample)} samples\n")

    for idx, row in val_sample.iterrows():
        clause = row['text']
        label = row['label']
        example_prompt = prompt_prefix + f"Clause: {clause}\nLabel:"

        try:
            if verbose:
                print(f"\n🔹 Prompt:\n{example_prompt[:200]}...")  # show first 200 chars

            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": example_prompt}],
                temperature=0,
            )
            prediction_raw = response.choices[0].message.content.strip()
            prediction = int(prediction_raw[0]) if prediction_raw[0] in ['0', '1'] else 0

            if verbose:
                print(f"🔸 Model response: '{prediction_raw}' → Parsed prediction: {prediction} | True label: {label}")

        except Exception as e:
            prediction = 0
            if verbose:
                print(f"❌ Error during prediction: {e}")

        preds.append(prediction)
        time.sleep(1.5)

    acc = accuracy_score(val_sample['label'], preds)

    if verbose:
        print(f"\n✅ Completed. Accuracy: {acc:.4f} on {len(preds)} examples.\n")

    return acc



In [32]:
import random

# Sample few-shot exemplars
exemplars = train_df.sample(5, random_state=42)

# Initialize instruction history
solutions = [("Classify if the clause is fair or unfair. Respond with 0 or 1 only.", 0.5)]

for step in range(10):
    meta_prompt = build_meta_prompt(exemplars, solutions[-5:])  # use last 5
    new_instruction = get_instruction(meta_prompt)
    print(f"Step {step + 1} - Generated: {new_instruction}")
    
    acc = score_instruction(new_instruction, exemplars, val_df)
    print(f"Validation Accuracy: {acc:.4f}")
    
    solutions.append((new_instruction, acc))
    time.sleep(2)  # avoid rate limits


Step 1 - Generated: Classify if the clause specifies enforceable legal actions for breaches of policies, guidelines, or rights. Respond with 1 if yes, 0 if no.

🧪 Scoring instruction: "Classify if the clause specifies enforceable legal actions for breaches of policies, guidelines, or rights. Respond with 1 if yes, 0 if no."
Using model: mistralai/mistral-7b-instruct on 20 samples


🔹 Prompt:
Classify if the clause specifies enforceable legal actions for breaches of policies, guidelines, or rights. Respond with 1 if yes, 0 if no.

Clause: section 4 : our intellectual property rights
Label:...
🔸 Model response: '1 (Yes)

This clause likely refers to the protection of intellectual property rights, which are enforceable by law. Breaches of these rights can lead to legal actions such as copyright infringement, trademark infringement, or patent infringement lawsuits.' → Parsed prediction: 1 | True label: 0

🔹 Prompt:
Classify if the clause specifies enforceable legal actions for breaches of 

In [35]:
print(solutions)

[('Classify if the clause is fair or unfair. Respond with 0 or 1 only.', 0.5), ('Classify if the clause specifies enforceable legal actions for breaches of policies, guidelines, or rights. Respond with 1 if yes, 0 if no.', 0.55), ('Classify if the clause specifies or describes legal consequences, obligations, or requirements related to the use of a service or product. Respond with 1 if yes, 0 if no.', 0.25), ('Classify if the clause includes clauses that outline the terms and conditions of the service or product, including consequences for non-compliance, enforceable legal actions for breaches, and/or requirements for the use of the service or product. Respond with 1 if yes, 0 if no.', 0.45), ('Classify if the clause specifies or describes legal consequences, obligations, or requirements that a user must adhere to in order to utilize a service or product, and if the clause includes enforceable legal actions for breaches. Respond with 1 if the clause contains both, 0 otherwise.', 0.5), 

In [ ]:
# Select best-performing instruction from OPRO
best_instruction = max(solutions, key=lambda x: x[1])[0]
print("🏆 Best Instruction:\n", best_instruction)

# Set the model to a cheap one on OpenRouter
model = "mistralai/mistral-7b-instruct"
preds = []

# Sample 500 rows randomly from the test set
test_sample = test_df.sample(n=500, random_state=42) if len(test_df) > 500 else test_df

print("\n🚀 Starting evaluation on test sample...")
print(f"Using model: {model}")
print(f"Total sampled examples: {len(test_sample)}\n")

# Evaluate best instruction on the sampled test set
for idx, row in test_sample.iterrows():
    clause = row['text']
    true_label = row['label']
    prompt = best_instruction + f"\n\nClause: {clause}\nLabel:"

    try:
        print(f"🔹 Example {idx + 1}/{len(test_sample)}")
        print(f"Prompt:\n{prompt[:300]}...\n")  # limit to 300 chars

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        prediction_raw = response.choices[0].message.content.strip()
        prediction = int(prediction_raw[0]) if prediction_raw[0] in ['0', '1'] else 0
        print(f"🔸 Model output: '{prediction_raw}' → Parsed: {prediction} | True: {true_label}\n")

    except Exception as e:
        prediction = 0
        print(f"❌ Error on example {idx}: {e}\n")

    preds.append(prediction)

# Compute and report metrics
acc = accuracy_score(test_sample['label'], preds)
f1 = f1_score(test_sample['label'], preds)

print(f"\n✅ Test Accuracy: {acc:.4f}")
print(f"✅ Test F1 Score: {f1:.4f}")


🏆 Best Instruction:
 Classify if the clause includes terms and conditions related to the use of a service or product for financial transactions, with a focus on clauses that outline the requirements for users, specify the evidence required for reporting potential fraud or disputes, and define the steps for resolution, including the potential involvement of legal actions and the specific rights and obligations of the parties involved, with an emphasis on credit card transactions and privacy-related concerns. Respond with 1 if the clause contains all of the above, 0 otherwise.

🚀 Starting evaluation on test set...
Using model: mistralai/mistral-7b-instruct
Total examples: 3784

🔹 Example 1/3784
Prompt:
Classify if the clause includes terms and conditions related to the use of a service or product for financial transactions, with a focus on clauses that outline the requirements for users, specify the evidence required for reporting potential fraud or disputes, and define the steps for res

KeyboardInterrupt: 